# Autoencodeur convolutif : réduire une carte mel et la reconstruire

Suivre les canaux et les tailles, puis comparer la reconstruction à une cible explicite.

Nous lisons et exécutons les étapes dans l’ordre. Les cellules « Observer » montrent les valeurs et les formes ; elles ne prouvent pas la réussite des exercices.

Environnement vérifié : Python 3.11.9, PyTorch 2.10.0 CPU et TorchAudio 2.10.0 lorsque le chapitre le demande. Aucun modèle ni jeu de données n’est téléchargé pendant l’exécution.

[Revenir au chapitre](https://radiocodeur.github.io/quiz/cours.html?theme=pytorch#conv-mel-autoencodeur).


## Préparer le noyau

Choisissez un environnement CPU neuf. En cas de bibliothèque manquante, installez-la dans ce noyau, puis redémarrez-le. Les sorties affichent les versions réellement utilisées.

In [1]:
import os
import sys
import tempfile

# Choisir un noyau neuf pour garantir la vérification sur CPU.
os.environ["CUDA_VISIBLE_DEVICES"] = ""
try:
    import torch
except ImportError as error:
    raise RuntimeError("PyTorch manque. Installer torch==2.10.0 dans ce noyau, puis redémarrer et relancer.") from error
torch.set_num_threads(1)
torch.manual_seed(17)
print("Python", sys.version.split()[0], "— PyTorch", torch.__version__, "— CPU")
assert not torch.cuda.is_available(), "Redémarrer dans un noyau CPU neuf."
_quiz_initial_directory = os.getcwd()
_quiz_temporary_directory = tempfile.TemporaryDirectory(prefix="quiz-progressif-")
os.chdir(_quiz_temporary_directory.name)


Python 3.11.9 — PyTorch 2.10.0+cpu — CPU


In [2]:
try:
    import torchaudio
except (ImportError, OSError) as error:
    raise RuntimeError("TorchAudio manque ou ne correspond pas à PyTorch. Installer torch==2.10.0 et torchaudio==2.10.0 ensemble, puis redémarrer.") from error
print("TorchAudio",torchaudio.__version__)

TorchAudio 2.10.0+cpu


In [3]:
# Cet outil sert uniquement à afficher les états des démonstrations.
# Il ne modifie pas les tenseurs et ne fait pas partie du modèle étudié.
def observer(nom, valeur):
    if isinstance(valeur, torch.Tensor):
        contenu = valeur.detach().cpu()
        print(nom, "forme", tuple(contenu.shape), "type", contenu.dtype,
              "requires_grad", valeur.requires_grad)
        if contenu.numel() <= 16:
            print("  valeurs :", contenu.tolist())
    elif valeur is None or isinstance(valeur, (int, float, bool, str, tuple)):
        print(nom, "=", valeur)
    elif isinstance(valeur, list) and len(valeur) <= 8:
        if all(isinstance(item,(int,float,bool,str)) for item in valeur):
            print(nom, "=", valeur)
        else:
            print(nom, ":",len(valeur),"éléments ; formes des tenseurs :",
                  [tuple(item.shape) for item in valeur if isinstance(item,torch.Tensor)])
    elif isinstance(valeur,dict):
        print(nom, ": clés",list(valeur)[:12])

def observer_entrainement(espace, montrer_poids=False):
    # Une copie évite de parcourir un dictionnaire global en mutation.
    vus = set()
    for nom, objet in list(espace.items()):
        if nom.startswith('_'):
            continue
        if isinstance(objet, torch.Tensor) and objet.is_leaf and objet.requires_grad:
            parametres = [(nom, objet)]
        elif isinstance(objet, torch.nn.Module):
            parametres = [(nom + '.' + cle, parametre)
                          for cle, parametre in list(objet.named_parameters())[:4]]
        else:
            continue
        for cle, parametre in parametres:
            if id(parametre) in vus:
                continue
            vus.add(id(parametre))
            if montrer_poids:
                observer(cle, parametre)
            observer(cle + '.grad', parametre.grad)


Nous savons préparer une entrée mel et calculer un petit filtre. Réunissons maintenant un encodeur qui réduit la carte et un décodeur qui tente de retrouver la même carte.

Ce programme autonome reprend exactement les deux sinusoïdes et la préparation expliquées précédemment. La découpe à 96 trames et la division par 100 sont donc les mêmes choix, pas de nouvelles règles du mel.

L’entrée est (2,1,64,96) : deux extraits, un canal de caractéristiques, 64 bandes et 96 trames. Les cartes logarithmiques sont notre cible de reconstruction.

| Étape | Canaux | Bandes | Trames |
| --- | --- | --- | --- |
| Entrée | 1 | 64 | 96 |
| Conv2d(1,4), stride 2 | 4 | 32 | 48 |
| Conv2d(4,8), stride 2 | 8 | 16 | 24 |
| ConvTranspose2d(8,4), stride 2 | 4 | 32 | 48 |
| ConvTranspose2d(4,1), stride 2 | 1 | 64 | 96 |

L’axe des deux extraits reste inchangé. Les canaux 4 et 8 sont des cartes de caractéristiques apprises, pas des classes.

Pour la première convolution : floor((64+2×1−3)/2)+1 = 32 bandes. Pour la première transposée : (16−1)×2−2×1+4 = 32.

Une convolution transposée apprend un réagrandissement. Elle ne connaît pas automatiquement les valeurs perdues par l’encodeur ; une forme compatible autorise seulement la comparaison.

## Démonstration 1 · Lire l’encodeur, le décodeur puis un pas d’apprentissage

Le prétraitement déjà expliqué reste présent pour que le programme soit autonome.

### Préparer les bibliothèques

Exécutez cette petite partie après la précédente. Si un objet est créé, la cellule d’observation suivante permet d’en examiner l’état.

In [4]:
import torch
import torchaudio


### Fixer les conditions du petit calcul

Exécutez cette petite partie après la précédente. Si un objet est créé, la cellule d’observation suivante permet d’en examiner l’état.

In [5]:
torch.manual_seed(17)


### Calculer l’état suivant

Les premières lignes fabriquent les deux ondes et leurs cartes. Les transformations sont celles du chapitre de préparation ; aucune analyse de fichier réel n’est cachée.

In [6]:
sr = 16000
temps = torch.arange(sr) / sr
onde_220 = 0.2 * torch.sin(2 * torch.pi * 220 * temps)
onde_440 = 0.2 * torch.sin(2 * torch.pi * 440 * temps)


In [7]:
observer('sr', sr)
observer('temps', temps)
observer('onde_220', onde_220)
observer('onde_440', onde_440)

sr = 16000
temps forme (16000,) type torch.float32 requires_grad False
onde_220 forme (16000,) type torch.float32 requires_grad False
onde_440 forme (16000,) type torch.float32 requires_grad False


### Calculer l’état suivant

Exécutez cette petite partie après la précédente. Si un objet est créé, la cellule d’observation suivante permet d’en examiner l’état.

In [8]:
ondes = torch.stack([onde_220, onde_440])
fenetre = torch.hann_window(400)
stft = torch.stft(ondes, n_fft=400, hop_length=160, window=fenetre,
                  center=False, return_complex=True)


In [9]:
observer('ondes', ondes)
observer('fenetre', fenetre)
observer('stft', stft)

ondes forme (2, 16000) type torch.float32 requires_grad False
fenetre forme (400,) type torch.float32 requires_grad False
stft forme (2, 201, 98) type torch.complex64 requires_grad False


### Calculer l’état suivant

Exécutez cette petite partie après la précédente. Si un objet est créé, la cellule d’observation suivante permet d’en examiner l’état.

In [10]:
transformation = torchaudio.transforms.MelSpectrogram(
    sample_rate=sr, n_fft=400, win_length=400, hop_length=160,
    n_mels=64, f_min=0., f_max=8000., center=False,
    mel_scale="slaney", norm="slaney", power=2.)
mel_brut = transformation(ondes)
mel = mel_brut / fenetre.sum().square()
mel_positif = mel.clamp_min(1e-10)


In [11]:
observer('transformation', transformation)
observer('mel_brut', mel_brut)
observer('mel', mel)
observer('mel_positif', mel_positif)

mel_brut forme (2, 64, 98) type torch.float32 requires_grad False
mel forme (2, 64, 98) type torch.float32 requires_grad False
mel_positif forme (2, 64, 98) type torch.float32 requires_grad False


### Calculer l’état suivant

Exécutez cette petite partie après la précédente. Si un objet est créé, la cellule d’observation suivante permet d’en examiner l’état.

In [12]:
mel_db = 10 * mel_positif.log10()
mel_reduit = mel_db / 100
mel_court = mel_reduit[:, :, :96]
entree = mel_court.unsqueeze(1)


In [13]:
observer('mel_db', mel_db)
observer('mel_reduit', mel_reduit)
observer('mel_court', mel_court)
observer('entree', entree)

mel_db forme (2, 64, 98) type torch.float32 requires_grad False
mel_reduit forme (2, 64, 98) type torch.float32 requires_grad False
mel_court forme (2, 64, 96) type torch.float32 requires_grad False
entree forme (2, 1, 64, 96) type torch.float32 requires_grad False


### Préparer les bibliothèques

Exécutez cette petite partie après la précédente. Si un objet est créé, la cellule d’observation suivante permet d’en examiner l’état.

In [14]:
from torch import nn


### Calculer l’état suivant

Sequential applique ses couches dans l’ordre. Le premier filtre lit un canal et en produit quatre ; le suivant lit ces quatre cartes et en produit huit.

In [15]:
encodeur = nn.Sequential(
    nn.Conv2d(1, 4, 3, stride=2, padding=1), nn.ReLU(),
    nn.Conv2d(4, 8, 3, stride=2, padding=1), nn.ReLU())


### Calculer l’état suivant

Les couches transposées doublent les deux tailles spatiales avec les paramètres choisis. La dernière couche produit un canal, comme la cible.

In [16]:
decodeur = nn.Sequential(
    nn.ConvTranspose2d(8, 4, 4, stride=2, padding=1), nn.ReLU(),
    nn.ConvTranspose2d(4, 1, 4, stride=2, padding=1))
modele = nn.Sequential(encodeur, decodeur)


### Calculer l’état suivant

La boucle sur encodeur, puis sur decodeur, remplace z par la sortie courante. enumerate fournit l’indice pour nommer chaque observation.

In [17]:
z = entree


In [18]:
observer('z', z)

z forme (2, 1, 64, 96) type torch.float32 requires_grad False


### Lire les sorties

Ces impressions lisent les objets calculés. Elles n’entraînent pas le modèle et ne modifient pas les tenseurs.

In [19]:
print("entree :", tuple(z.shape))


entree : (2, 1, 64, 96)


### Répéter la même étape sur les données

Le corps indenté reste dans une seule cellule. Les opérations se répètent dans l’ordre écrit ; les sorties permettront de suivre leurs effets.

In [20]:
for indice, couche in enumerate(encodeur):
    z = couche(z)
    print("encodeur", indice, tuple(z.shape))


encodeur 0 (2, 4, 32, 48)
encodeur 1 (2, 4, 32, 48)
encodeur 2 (2, 8, 16, 24)
encodeur 3 (2, 8, 16, 24)


### Répéter la même étape sur les données

Le corps indenté reste dans une seule cellule. Les opérations se répètent dans l’ordre écrit ; les sorties permettront de suivre leurs effets.

In [21]:
for indice, couche in enumerate(decodeur):
    z = couche(z)
    print("decodeur", indice, tuple(z.shape))


decodeur 0 (2, 4, 32, 48)
decodeur 1 (2, 4, 32, 48)
decodeur 2 (2, 1, 64, 96)


### Choisir les paramètres et leur mise à jour

reconstruction est produite par le modèle ; entree est la cible. MSELoss moyenne les carrés de leurs différences sur toutes les valeurs.

In [22]:
critere = nn.MSELoss()
optimiseur = torch.optim.Adam(modele.parameters(), lr=0.01)


### Remettre les gradients à zéro

Exécutez cette petite partie après la précédente. Si un objet est créé, la cellule d’observation suivante permet d’en examiner l’état.

In [23]:
optimiseur.zero_grad()


In [24]:
observer_entrainement({'sr': sr, 'temps': temps, 'onde_220': onde_220, 'onde_440': onde_440, 'ondes': ondes, 'fenetre': fenetre, 'stft': stft, 'transformation': transformation, 'mel_brut': mel_brut, 'mel': mel, 'mel_positif': mel_positif, 'mel_db': mel_db, 'mel_reduit': mel_reduit, 'mel_court': mel_court, 'entree': entree, 'encodeur': encodeur, 'decodeur': decodeur, 'modele': modele, 'z': z, 'critere': critere, 'optimiseur': optimiseur}, montrer_poids=False)

encodeur.0.weight.grad = None
encodeur.0.bias.grad = None
encodeur.2.weight.grad = None
encodeur.2.bias.grad = None
decodeur.0.weight.grad = None
decodeur.0.bias.grad = None
decodeur.2.weight.grad = None
decodeur.2.bias.grad = None


### Calculer l’état suivant

Exécutez cette petite partie après la précédente. Si un objet est créé, la cellule d’observation suivante permet d’en examiner l’état.

In [25]:
reconstruction = modele(entree)
perte = critere(reconstruction, entree)
avant = perte.item()


In [26]:
observer('reconstruction', reconstruction)
observer('perte', perte)
observer('avant', avant)

reconstruction forme (2, 1, 64, 96) type torch.float32 requires_grad True
perte forme () type torch.float32 requires_grad True
  valeurs : 0.8759986758232117
avant = 0.8759986758232117


### Calculer les gradients

backward remonte dans les deux parties. Adam les met à jour ; no_grad sert ensuite à observer la nouvelle perte sans préparer une autre dérivation.

In [27]:
perte.backward()


In [28]:
observer_entrainement({'sr': sr, 'temps': temps, 'onde_220': onde_220, 'onde_440': onde_440, 'ondes': ondes, 'fenetre': fenetre, 'stft': stft, 'transformation': transformation, 'mel_brut': mel_brut, 'mel': mel, 'mel_positif': mel_positif, 'mel_db': mel_db, 'mel_reduit': mel_reduit, 'mel_court': mel_court, 'entree': entree, 'encodeur': encodeur, 'decodeur': decodeur, 'modele': modele, 'z': z, 'critere': critere, 'optimiseur': optimiseur, 'reconstruction': reconstruction, 'perte': perte, 'avant': avant}, montrer_poids=False)

encodeur.0.weight.grad forme (4, 1, 3, 3) type torch.float32 requires_grad False
encodeur.0.bias.grad forme (4,) type torch.float32 requires_grad False
  valeurs : [-0.0020042110700160265, 0.013977506197988987, 0.023997362703084946, 0.00016853325359988958]
encodeur.2.weight.grad forme (8, 4, 3, 3) type torch.float32 requires_grad False
encodeur.2.bias.grad forme (8,) type torch.float32 requires_grad False
  valeurs : [0.0002566033217590302, -0.0025012814439833164, 0.0, 0.0, 0.007029403932392597, 0.0, 0.02288668043911457, 0.01873181201517582]
decodeur.0.weight.grad forme (8, 4, 4, 4) type torch.float32 requires_grad False
decodeur.0.bias.grad forme (4,) type torch.float32 requires_grad False
  valeurs : [-0.14298169314861298, 0.13798876106739044, 0.035217031836509705, 0.053063664585351944]
decodeur.2.weight.grad forme (4, 1, 4, 4) type torch.float32 requires_grad False
decodeur.2.bias.grad forme (1,) type torch.float32 requires_grad False
  valeurs : [1.845211386680603]


### Appliquer une mise à jour

Exécutez cette petite partie après la précédente. Si un objet est créé, la cellule d’observation suivante permet d’en examiner l’état.

In [29]:
optimiseur.step()


In [30]:
observer_entrainement({'sr': sr, 'temps': temps, 'onde_220': onde_220, 'onde_440': onde_440, 'ondes': ondes, 'fenetre': fenetre, 'stft': stft, 'transformation': transformation, 'mel_brut': mel_brut, 'mel': mel, 'mel_positif': mel_positif, 'mel_db': mel_db, 'mel_reduit': mel_reduit, 'mel_court': mel_court, 'entree': entree, 'encodeur': encodeur, 'decodeur': decodeur, 'modele': modele, 'z': z, 'critere': critere, 'optimiseur': optimiseur, 'reconstruction': reconstruction, 'perte': perte, 'avant': avant}, montrer_poids=True)

encodeur.0.weight forme (4, 1, 3, 3) type torch.float32 requires_grad True
encodeur.0.weight.grad forme (4, 1, 3, 3) type torch.float32 requires_grad False
encodeur.0.bias forme (4,) type torch.float32 requires_grad True
  valeurs : [0.19888471066951752, 0.14376017451286316, -0.046687014400959015, -0.15152062475681305]
encodeur.0.bias.grad forme (4,) type torch.float32 requires_grad False
  valeurs : [-0.0020042110700160265, 0.013977506197988987, 0.023997362703084946, 0.00016853325359988958]
encodeur.2.weight forme (8, 4, 3, 3) type torch.float32 requires_grad True
encodeur.2.weight.grad forme (8, 4, 3, 3) type torch.float32 requires_grad False
encodeur.2.bias forme (8,) type torch.float32 requires_grad True
  valeurs : [-0.16481776535511017, 0.006333737634122372, 0.09473259747028351, -0.15995699167251587, 0.031158989295363426, -0.07432162761688232, -0.06967157870531082, 0.01636720821261406]
encodeur.2.bias.grad forme (8,) type torch.float32 requires_grad False
  valeurs : [0.000256603

### Appliquer le contexte explicite

Exécutez cette petite partie après la précédente. Si un objet est créé, la cellule d’observation suivante permet d’en examiner l’état.

In [31]:
with torch.no_grad():
    apres = critere(modele(entree), entree).item()


### Lire les sorties

Ces impressions lisent les objets calculés. Elles n’entraînent pas le modèle et ne modifient pas les tenseurs.

In [32]:
print(f"MSE avant={avant:.6f}, apres={apres:.6f}")

MSE avant=0.875999, apres=0.796794


La diminution observée vérifie qu’un pas agit sur ces deux entrées. Elle ne démontre pas une reconstruction audio utile ou une généralisation.

## À vous de modifier le calcul

Les deux cellules suivantes sont prêtes à éditer. Si vous les exécutez sans les compléter, elles le signalent et ne prétendent pas que vous avez réussi. Une réponse fausse déclenche le contrôle correspondant.

### Exercice 1

Conv2d(1,8,3,stride=2,padding=1) reçoit (2,1,64,96). Quelle forme produit-elle ?

In [33]:
exo_x = torch.zeros(2,1,64,96)
exo_conv = torch.nn.Conv2d(1,8,3,stride=2,padding=1)
exo_forme = ...  # Remplacer les points par votre réponse.
if exo_forme is Ellipsis or exo_forme is None:
    print("À compléter : cet exercice n’est pas encore réalisé.")
else:
    assert tuple(exo_forme) == tuple(exo_conv(exo_x).shape)
    print("Contrôle réussi pour la réponse fournie. Expliquez maintenant pourquoi.")


À compléter : cet exercice n’est pas encore réalisé.


<details>
<summary>Indice pour l’exercice 1</summary>

Le lot reste 2, les canaux deviennent 8, et les deux tailles spatiales sont divisées par deux ici.

</details>

### Exercice 2

Une reconstruction [1,2] est comparée à une cible [0,0] par MSE moyenne. Quelle perte ?

In [34]:
exo_reconstruction = torch.tensor([1.,2.])
exo_cible = torch.zeros(2)
exo_perte = ...  # Remplacer les points par votre réponse.
if exo_perte is Ellipsis or exo_perte is None:
    print("À compléter : cet exercice n’est pas encore réalisé.")
else:
    assert abs(float(exo_perte)-torch.nn.functional.mse_loss(exo_reconstruction,exo_cible).item()) < 1e-6
    print("Contrôle réussi pour la réponse fournie. Expliquez maintenant pourquoi.")


À compléter : cet exercice n’est pas encore réalisé.


<details>
<summary>Indice pour l’exercice 2</summary>

La moyenne des erreurs au carré vaut (1+4)/2. Ce n’est pas la moyenne des valeurs.

</details>

## Comparer avec les solutions

Ouvrez-les après avoir formulé votre réponse et votre explication. Les solutions sont également exécutées par le vérificateur du dépôt.

<details>
<summary>Solution de l’exercice 1</summary>

```python
exo_x = torch.zeros(2,1,64,96)
exo_conv = torch.nn.Conv2d(1,8,3,stride=2,padding=1)
exo_forme = (2,8,32,48)
if exo_forme is Ellipsis or exo_forme is None:
    print("À compléter : cet exercice n’est pas encore réalisé.")
else:
    assert tuple(exo_forme) == tuple(exo_conv(exo_x).shape)
    print("Contrôle réussi pour la réponse fournie. Expliquez maintenant pourquoi.")
```

</details>

<details>
<summary>Solution de l’exercice 2</summary>

```python
exo_reconstruction = torch.tensor([1.,2.])
exo_cible = torch.zeros(2)
exo_perte = 2.5
if exo_perte is Ellipsis or exo_perte is None:
    print("À compléter : cet exercice n’est pas encore réalisé.")
else:
    assert abs(float(exo_perte)-torch.nn.functional.mse_loss(exo_reconstruction,exo_cible).item()) < 1e-6
    print("Contrôle réussi pour la réponse fournie. Expliquez maintenant pourquoi.")
```

</details>

In [35]:
os.chdir(_quiz_initial_directory)
_quiz_temporary_directory.cleanup()